## Step 1: Install feast, scikit-learn

Install feast, gcp dependencies and scikit-learn


In [ ]:
!pip install feast scikit-learn 'feast[gcp]' google

#### Check feast version

In [3]:
!feast version

Feast SDK Version: "0.54.0"


## Step 2: Set up your Goggle Cloud Platform (GCP) Configurations

In [5]:
PROJECT_ID= "mystical-atlas-473806-i7" #@param {type:"string"}
BUCKET_NAME= "mystical-atlas-473806-i7-driver_ranking_tutorial" #@param {type:"string"} custom
BIGQUERY_DATASET_NAME="iris_model" #@param {type:"string"} custom
AI_PLATFORM_MODEL_NAME="iris_jsd_model" #@param {type:"string"

! gcloud config set project $PROJECT_ID
%env GOOGLE_CLOUD_PROJECT=$PROJECT_ID
!echo project_id = $PROJECT_ID > ~/.bigqueryrc

Updated property [core/project].
env: GOOGLE_CLOUD_PROJECT=mystical-atlas-473806-i7


In [4]:
# Only run if your bucket doesn't already exist!
! gsutil mb gs://$BUCKET_NAME

Creating gs://mystical-atlas-473806-i7-driver_ranking_tutorial/...
ServiceException: 409 A Cloud Storage bucket named 'mystical-atlas-473806-i7-driver_ranking_tutorial' already exists. Try another name. Bucket names must be globally unique across all Google Cloud projects, including those outside of your organization.


## Step 4: Apply and deploy feature definitions

`feast apply` scans python files in the current directory for feature definitions and deploys infrastructure according to `feature_store.yaml`

In [10]:
%%bash
cd feature_store; feast apply

bash: line 2: cd: feature_store: No such file or directory


No project found in the repository. Using project name iris defined in feature_store.yaml
Applying changes for project iris
Deploying infrastructure for iris_measurements_v1


### Inspect the files created under your local folder

In [11]:
%%bash
cd feature_store/data; ls -la

total 28
drwxr-xr-x 2 jupyter jupyter  4096 Oct 11 19:08 .
drwxr-xr-x 5 jupyter jupyter  4096 Oct 11 12:53 ..
-rw-r--r-- 1 jupyter jupyter 16384 Oct 11 19:08 online.db
-rw-r--r-- 1 jupyter jupyter   911 Oct 11 19:08 registry.db


## Step 5: Train your model

In [13]:
!pip install numpy pandas

In [14]:
import feast
import pandas as pd
from sklearn.linear_model import LogisticRegression
from sklearn.preprocessing import LabelEncoder
from joblib import dump

fs = feast.FeatureStore(repo_path="feature_store")

full_data = pd.read_csv("iris_data_adapted_for_feast.csv")
full_data["event_timestamp"] = pd.to_datetime(full_data["event_timestamp"])
full_data["iris_id"] = full_data["iris_id"].astype('Int64')

entity_df = full_data[["event_timestamp", "iris_id"]]

training_df = fs.get_historical_features(
    entity_df=entity_df,
    features=[
        "iris_measurements_v1:sepal_length",
        "iris_measurements_v1:sepal_width",
        "iris_measurements_v1:petal_length",
        "iris_measurements_v1:petal_width",
        "iris_measurements_v1:species",
    ],
).to_df()

print("--- Feature Schema (Training Data) ---\n")
print(training_df.info())

print("\n--- Example Features ---\n")
print(training_df.head())

le = LabelEncoder()
target = "species"
target_encoded = "species_encoded"

features_to_drop = [target, "event_timestamp", "iris_id"]
train_X = training_df.drop(columns=features_to_drop, errors='ignore')
train_Y = le.fit_transform(training_df[target])

reg = LogisticRegression(max_iter=500)
reg.fit(train_X, train_Y)

dump(reg, "iris_classifier.bin")

print("\n--- Training Complete ---")
print("Model 'iris_classifier.bin' saved successfully.")


--- Feature Schema (Training Data) ---

<class 'pandas.core.frame.DataFrame'>
RangeIndex: 45 entries, 0 to 44
Data columns (total 7 columns):
 #   Column           Non-Null Count  Dtype              
---  ------           --------------  -----              
 0   event_timestamp  45 non-null     datetime64[us, UTC]
 1   iris_id          45 non-null     int64              
 2   sepal_length     45 non-null     float64            
 3   sepal_width      45 non-null     float64            
 4   petal_length     45 non-null     float64            
 5   petal_width      45 non-null     float64            
 6   species          45 non-null     object             
dtypes: datetime64[us, UTC](1), float64(4), int64(1), object(1)
memory usage: 2.6+ KB
None

--- Example Features ---

                   event_timestamp  iris_id  sepal_length  sepal_width  \
0 2025-09-17 10:40:17.102131+00:00     1001          5.52         2.53   
1 2025-09-17 10:40:17.102131+00:00     1003          5.09         3.42

## Step 6: Materialize your online store
Apply and materialize data to Firestore

In [16]:
%%bash
cd feature_store

START_DATE="2025-09-01T00:00:00"
END_DATE="2025-10-31T23:59:59"

# Materialize all features for the given time window into the online store (SQLite)
feast materialize ${START_DATE} ${END_DATE}

Materializing 1 feature views from 2025-09-01 00:00:00+00:00 to 2025-10-31 23:59:59+00:00 into the sqlite online store.

iris_measurements_v1:


### Step 7:  Make Prediction

In [30]:
class IrisClassifier:
    def __init__(self):
        # Load model only
        self.model = load("iris_classifier.bin") 

        # Set up feature store
        # Assumes feature_repo is in the current working directory
        self.fs = feast.FeatureStore(repo_path="feature_store")

        # Define feature list (model input order matters!)
        self.feature_list = [
            "iris_measurements_v1:sepal_length",
            "iris_measurements_v1:sepal_width",
            "iris_measurements_v1:petal_length",
            "iris_measurements_v1:petal_width",
        ]

        # Define the short feature names used by the model
        self.model_feature_names = [f.split(":")[-1] for f in self.feature_list]

    def predict(self, iris_ids):
        # Read features from Feast Online Store
        entity_rows = [{"iris_id": id} for id in iris_ids]

        online_features = self.fs.get_online_features(
            entity_rows=entity_rows,
            features=self.feature_list,
        )
        online_df = online_features.to_df()

        # Define the actual feature columns returned by Feast (which are the short names in your current setup)
        feature_columns_in_df = self.model_feature_names
        
        # 1. Select the feature columns for prediction (X_predict)
        X_predict = online_df[feature_columns_in_df]
        
        # 2. Ensure X_predict columns match the training order (essential for correct prediction)
        X_predict = X_predict[self.model_feature_names]

        # Make prediction (outputs integer class IDs: 0, 1, 2)
        predictions_int = self.model.predict(X_predict)
        
        # Build results DataFrame
        results = online_df.copy()
        # Changed column name to reflect the raw integer output
        results['predicted_class_id'] = predictions_int

        # Return the required columns
        return results[['iris_id', 'predicted_class_id'] + feature_columns_in_df]


In [31]:
# --- Prediction Execution ---

# Instantiate the classifier service
classifier_service = IrisClassifier()

# Define the iris IDs to retrieve features for
# These IDs must exist in your materialized online store
iris_ids_to_predict = [1001, 1002, 1003] 

print("--- Retrieving Online Features for 3 Iris IDs ---")

--- Retrieving Online Features for 3 Iris IDs ---


In [32]:
# Get predictions
final_predictions_df = classifier_service.predict(iris_ids_to_predict)

print("\n--- Final Predictions Output ---")
print(final_predictions_df)


--- Final Predictions Output ---
   iris_id  predicted_class_id  sepal_length  sepal_width  petal_length  \
0     1001                   1          5.45         2.36          3.84   
1     1002                   0          4.84         2.90          1.29   
2     1003                   0          4.85         3.40          1.19   

   petal_width  
0         1.09  
1         0.20  
2         0.29  
